# Golden Model for NMS Algorithm

This notebook is the **narrative** for the NMS golden model. The algorithm itself lives in
`models/nms/model.py` and this notebook imports it, so there is exactly one implementation
rather than two that can drift apart.

The reference implementation is verified by `models/nms/test_model.py`:

- both algorithm forms reproduce `keep_mask = 0xE1010101` on the test set below,
- they agree over 20,000 adversarial batches (20.5 million predicate evaluations),
- no division is performed anywhere,
- the union can never underflow.

Run `uv run pytest models/nms/` to check all of that.

In [ ]:
import sys
from pathlib import Path

# Make the repository root importable however this notebook was launched: Jupyter puts the
# notebook's own directory on sys.path, so `models.nms` would not resolve when started from
# models/. Walk up to the directory holding pyproject.toml.
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## What a box looks like

Each detection is five numbers:

```
(x, y, a, b, confidence)
```

- `(x, y)` — lower-left corner
- `(a, b)` — upper-right corner
- `confidence` — how sure the detector is, 0 to 1

In hardware each box is a 64-bit record: four 12-bit coordinates and a 16-bit score, with the
confidence quantised to `round(f * 65535)`. The layout is frozen in
[docs/architecture.md](../docs/architecture.md).

The 12 initial representation of coodinates comes from UART bit alignment. Take 12 bit for granted and do the calculations

Bounding box rejection criteria
$$\text{Area\_Overlap} \times 2^k \geq \text{Threshold\_INT} \times Area\_Union$$

| variable | Signed / Unsigned | Integer Bits | Fractional Bits | Total Width | Domain max (1080p) | Representable Range |
| --- | --- | --- | --- | --- | --- | --- |
| x/y/a/b | u | 12 | 0 | 12 | 1920 | 0 to 4095 |
| area1/area2 | u | 24 | 0 | 24 | 2073600 | 0 to 16777215 |
| xx/yy/aa/bb | u | 12 | 0 | 12 | 1920 | 0 to 4095 |
| t_w/t_h | s | 13 | 0 | 13 | 1920 | -4096 to +4095
| w/h | u | 12 | 0 | 12 | 1920 | 0 to 4095 | # can clamp it back
| intersection_area | u | 24 | 0 | 24 | 2073600 | 0 to 16777215 |
|  t_union_area | s | 26 | 0 | 26 | 2073600 | -33554432 to 33554431 | # positive side dominates thus to account for sign, we need 26 bits (assumes that the 3 data paths are independent)
|  union_area | u | 25 | 0 | 25 | 2073600 | 0 to 33554431 | # can clamp it back to 25 bits
|  T_INT | u | 8 | 0 | 8 | 126 | 0 to 126 |
|  2^k | u | 8 | 0 | 8 | 126 | 0 to 126 | # Q0.8 precision is chosen for simplicity
|  LHS | u | 32 | 0 | 32 | 261273600 | 0 to 4294967296 |
|  RHS | u | 33 | 0 | 33 | 261273600 | 0 to 8,589,934,592 | # 33 bits would be fixed length to compare

## Step 1: measuring overlap with IoU

**IoU (Intersection over Union)** measures how much two boxes overlap — 0 for no overlap, 1 for
identical boxes:

```
IoU = (area where the boxes overlap) / (total area covered by both)
```

Worked through by hand on two boxes, to show the arithmetic before it gets wrapped up:

In [ ]:
x1, y1, a1, b1, c1 = 200, 300, 400, 500, 0.86  # box 1
x2, y2, a2, b2, c2 = 350, 350, 450, 600, 0.65  # box 2

area1 = (a1 - x1) * (b1 - y1)
area2 = (a2 - x2) * (b2 - y2)

# the intersection maximises the lower-left and minimises the upper-right
xx, yy = max(x1, x2), max(y1, y2)
aa, bb = min(a1, a2), min(b1, b2)

w = max(0, aa - xx)  # clamped: a miss must give 0, not a negative width
h = max(0, bb - yy)

intersection_area = w * h
union_area = area1 + area2 - intersection_area

print(f"I = {intersection_area}, U = {union_area}, IoU = {intersection_area / union_area:.4f}")

## Step 2: why the hardware does not compute that ratio

The division above is convenient in Python and expensive in hardware. Instead of computing IoU
and comparing it to a threshold, the predicate is **cross-multiplied**:

```
suppress  when   I * 2^k  >=  T_INT * U
```

with `k = 8` and `T_INT = 128`, i.e. a threshold of `128/256 = 0.5`. Three consequences:

1. **No divider.** At `T_INT = 128` it reduces to `2I >= U` — two shifts and a compare.
2. **The boundary is exact.** A pair sitting precisely on `2I == U` is decided by integer
   comparison, not by whichever way a float division happened to round.
3. **No divide-by-zero.** Two degenerate boxes give `I = 0, U = 0`. The ratio form *crashes*
   here — the `degenerate` test case contains 49 such pairs — while the integer form gives
   `0 >= 0` and suppresses. So cross-multiplying is not merely a hardware convenience; it
   removes a defect that was present in the original version of this notebook.

The same predicate the RTL evaluates:

In [ ]:
from models.nms import model, params

box1 = model.Box(x1, y1, a1, b1, params.quantise_score(c1))
box2 = model.Box(x2, y2, a2, b2, params.quantise_score(c2))

print(f"areas         {model.box_area(box1)}, {model.box_area(box2)}")
print(f"intersection  {model.intersection_area(box1, box2)}")
print(f"suppresses    {model.suppresses(box1, box2)}")
print(f"threshold     T_INT={params.T_INT}/2**{params.K_SHIFT} = {params.T_INT / (1 << params.K_SHIFT)}")

# a degenerate pair: defined here, a ZeroDivisionError for the ratio form
zero = model.Box(100, 100, 100, 100, 1000)
print(f"degenerate    I=0, U=0 -> suppresses={model.suppresses(zero, zero)}")

## Step 3: the NMS algorithm

Given a list of boxes and a threshold, decide which to keep:

1. **Sort by confidence**, most confident first.
2. **Walk the list.** Each box not already discarded is kept as the winner for its object.
3. **Suppress overlaps** — every remaining box with IoU >= threshold against that winner is a
   duplicate and is discarded.

`model.py` provides this twice, on purpose:

- **`nms_sequential`** — the loop above. The authority on what NMS *means*.
- **`nms_allpairs`** — every pair evaluated first, then resolved in rank order. This is what the
  hardware implements, because it lets 16 lanes work in parallel and pays the pipeline drain
  once rather than once per keeper.

Keeping both and asserting they agree is what stops a mistake in the *restructuring* from
passing unnoticed — a single implementation would only ever be checked against itself.

### Tie-breaking

Python's `sorted` is stable, so equal scores keep input order. A bitonic sorting network is
**not** stable. Rather than making the network stable, ties are made impossible: the sort key is

```
K = score * 32 + (31 - index)
```

Indices are unique, so `K` is a strict total order and the network's instability can never be
observed — at zero hardware cost.

## Step 4: trying it on realistic data

32 boxes shaped like a detector's raw output: three clusters of ~8 heavily overlapping boxes
(cat, dog, car), a smaller cluster of 5 (a person), and a few isolated boxes that overlap
nothing and should always survive.

This list is the **source of truth** for the test set. `models/nms/batches.py` carries a
quantised copy, and `test_model.py` re-executes this notebook to assert the two still match.

In [ ]:
# (x1, y1, x2, y2, confidence)
test_boxes = [
    # Cluster A — cat detection (~8 overlapping boxes)
    (10, 10, 50, 50, 0.95),
    (12, 11, 52, 51, 0.90),
    (9, 13, 48, 53, 0.85),
    (14, 9, 54, 49, 0.70),
    (11, 14, 51, 54, 0.65),
    (15, 12, 55, 52, 0.55),
    (8, 8, 46, 46, 0.40),
    (13, 15, 53, 55, 0.30),
    # Cluster B — dog detection (~8 overlapping boxes)
    (100, 100, 150, 150, 0.92),
    (102, 101, 152, 151, 0.88),
    (98, 103, 148, 153, 0.80),
    (104, 99, 154, 149, 0.72),
    (101, 105, 151, 155, 0.60),
    (103, 98, 153, 148, 0.50),
    (97, 102, 147, 152, 0.42),
    (105, 104, 155, 154, 0.28),
    # Cluster C — car detection (~8 overlapping boxes)
    (200, 50, 260, 100, 0.93),
    (202, 52, 262, 102, 0.87),
    (198, 48, 258, 98, 0.78),
    (204, 53, 264, 103, 0.68),
    (201, 47, 261, 97, 0.58),
    (199, 55, 259, 105, 0.48),
    (203, 49, 263, 99, 0.38),
    (197, 51, 257, 101, 0.25),
    # Cluster D — person detection (~5 overlapping boxes)
    (50, 200, 100, 280, 0.91),
    (52, 202, 102, 282, 0.82),
    (48, 198, 98, 278, 0.73),
    (54, 203, 104, 283, 0.62),
    (47, 199, 97, 279, 0.45),
    # Isolated boxes — should all survive
    (300, 300, 340, 340, 0.75),
    (0, 280, 30, 310, 0.35),
    (280, 0, 320, 40, 0.20),
]

In [ ]:
from models.nms import batches

boxes = [
    model.Box(x, y, a, b, params.quantise_score(c)) for x, y, a, b, c in test_boxes
]

sequential = model.nms_sequential(boxes)
allpairs = model.nms_allpairs(boxes)
survivors = [i for i in range(len(boxes)) if sequential >> i & 1]

print(f"boxes in         {len(boxes)}")
print(f"survivors        {len(survivors)} at slots {survivors}")
print(f"keep_mask        0x{sequential:08X}")
print(f"both forms agree {sequential == allpairs}")
print(f"matches anchor   {sequential == batches.NOTEBOOK_KEEP_MASK}")

## What this set does and does not test

It reduces 32 boxes to 7 — one winner per cluster plus the isolated boxes — which confirms the
algorithm collapses duplicates while leaving unrelated boxes alone.

But it contains **no duplicate scores and no pair exactly on the threshold**, so it exercises
neither tie-breaking nor the boundary predicate: the two subtlest parts of the design. Passing
it alone proves very little.

That is why `models/nms/batches.py` adds `ties`, `all_equal`, `boundary`, `degenerate`,
`disjoint`, `all_survive`, `low_res_scores` and ten random seeds. Each targets something this
set cannot reach, and `models/nms/vectors.py` writes them out as the files the VHDL testbenches
read.

## Where this fits

| | |
|---|---|
| [docs/architecture.md](../docs/architecture.md) | the frozen spec — record format, widths, predicate |
| [docs/plan.md](../docs/plan.md) | the build plan and its gates |
| [docs/build_log.md](../docs/build_log.md) | what each gate actually measured |
| `models/nms/model.py` | this algorithm, in integer form |
| `models/data/vectors/` | the generated testbench vectors |

Once the RTL exists, its output is compared **bit-exactly** against this model — a single 32-bit
`keep_mask` equality, with no tolerance band.